# Relatório final: previsão do risco de violação do SLA de entrega no Olist

Este relatório reúne, em um único documento, o que foi feito e o que foi encontrado no projeto. A pergunta é se o risco de um pedido ser entregue depois do dia prometido pode ser previsto no momento da compra, com um desempenho superior ao de regras simples. Todos os números e gráficos do documento são lidos de resultados já gravados em `outputs/`: o torneio de modelos (V1), a ablação da feature de distância, os estudos do Optuna, as avaliações F1 e F2 e o ledger do período de teste. Nenhuma célula treina modelo, abre o período de teste ou recalcula uma métrica de modelo.

Quatro afirmações de estado acompanham qualquer leitura deste relatório. Primeira, o F1 é o resultado oficial, avaliado uma única vez conforme o pré-registro P-05 (Nosek et al., 2018) de `docs/REGISTRO-DE-DECISOES.md`, e foi inconclusivo. Segunda, o F2 é uma análise complementar pós-hoc e exploratória, desenhada depois de o período do F1 ter sido observado, e não confirma nem refuta o F1. Terceira, julho e agosto de 2018 foram consultados no F1 e no F2 e estão esgotados para novas avaliações. Quarta, `production_eligible = false`: nenhum modelo está habilitado para uso operacional.

## 1. Pergunta e escopo

A pergunta é aplicada: com as informações disponíveis no momento da compra, é possível identificar os pedidos de maior risco de chegar depois do dia prometido, e esse sinal supera regras simples, como o próprio prazo prometido? O alvo principal é binário, a violação do SLA por dia de calendário, e o alvo secundário é o atraso em dias, tratado de forma descritiva. O escopo é o conjunto público de pedidos do Olist (Olist, 2018), restrito a pedidos entregues. O conjunto não traz dados de capacidade, de tráfego ou de rota real, e por isso o relatório não avalia nenhuma decisão operacional.

A célula seguinte apenas carrega os resultados gravados que sustentam o relatório. Ela mostra a procedência de cada um, o ledger das avaliações do período de teste e o campo `production_eligible`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

LANG = "pt"
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "data").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from ecom_delivery_sla import describe_run, load_run_unchecked

CACHE_ROOT = project_root / "outputs" / "runs"
OPTUNA_DB = project_root / "outputs" / "optuna" / "sla_optimization.db"
STUDY_VERSION = "V4_folds"

LABELS = {
    "pt": {
        "post_hoc": "Execução pós-hoc (exploratória): F1 = {f1}; F2 = {f2}",
        "eligible": "production_eligible = {value}",
        "reading": "Leitura pré-registrada da diferença pareada de AP (CatBoost - baseline logística): {outcome}",
        "discarded": "Reamostragens descartadas por conter uma só classe: {n} de {total}",
        "all_orders": "Pedidos na tabela de pedidos",
        "supervised": "Pedidos supervisionados (entregues, com datas válidas)",
        "period": "Compras de {first} a {last}",
        "baseline": "baseline obrigatória",
        "model": "modelo candidato",
        "task": {"classification": "classificação", "regression": "regressão"},
        "protocol": {
            "first_val_month": "Primeiro mês de validação",
            "last_val_month": "Último mês de validação",
            "embargo_days": "Embargo entre treino e validação (dias)",
            "random_state": "Semente",
            "n_folds": "Folds de validação",
        },
    },
    "en": {
        "post_hoc": "Post-hoc (exploratory) run: F1 = {f1}; F2 = {f2}",
        "eligible": "production_eligible = {value}",
        "reading": "Pre-registered reading of the paired AP difference (CatBoost - logistic baseline): {outcome}",
        "discarded": "Resamples discarded for containing a single class: {n} of {total}",
        "all_orders": "Orders in the orders table",
        "supervised": "Supervised orders (delivered, with valid dates)",
        "period": "Purchases from {first} to {last}",
        "baseline": "mandatory baseline",
        "model": "candidate model",
        "task": {"classification": "classification", "regression": "regression"},
        "protocol": {
            "first_val_month": "First validation month",
            "last_val_month": "Last validation month",
            "embargo_days": "Embargo between train and validation (days)",
            "random_state": "Seed",
            "n_folds": "Validation folds",
        },
    },
}[LANG]

tournament, tournament_meta = load_run_unchecked("tournament", "V1", cache_root=CACHE_ROOT)
ablation, ablation_meta = load_run_unchecked("distance_ablation", "V1", cache_root=CACHE_ROOT)
f1, f1_meta = load_run_unchecked("final_evaluation", "F1", cache_root=CACHE_ROOT)
f2, f2_meta = load_run_unchecked("final_evaluation", "F2", cache_root=CACHE_ROOT)
for meta in (tournament_meta, ablation_meta, f1_meta, f2_meta):
    print(describe_run(meta, language=LANG))

print(
    LABELS["post_hoc"].format(
        f1=bool(f1["provenance"]["post_hoc"].iloc[0]),
        f2=bool(f2["provenance"]["post_hoc"].iloc[0]),
    )
)
ledger_path = CACHE_ROOT / "final_evaluation" / "LEDGER.jsonl"
display(pd.read_json(ledger_path, lines=True, dtype=False, convert_dates=False))
production_eligible = bool(f2["provenance"]["production_eligible"].iloc[0])
print(LABELS["eligible"].format(value=production_eligible))

Resultados carregados de outputs/runs/tournament/V1 (gerados em 2026-11-16 12:48, 97,6 min de treino; fingerprint c98673de)
Resultados carregados de outputs/runs/distance_ablation/V1 (gerados em 2026-11-16 13:01, 12,7 min de treino; fingerprint 18accc1f)
Resultados carregados de outputs/runs/final_evaluation/F1 (gerados em 2026-11-14 10:53, 1,3 min de treino; fingerprint ddc0ef97)
Resultados carregados de outputs/runs/final_evaluation/F2 (gerados em 2026-11-14 12:02, 16,3 min de treino; fingerprint a6fd373f)
Execução pós-hoc (exploratória): F1 = False; F2 = True


,version,timestamp,fingerprint,train_orders,test_orders,post_hoc,n_boot,seed,gate_passed
0,F1,2026-11-14T10:53:17,ddc0ef97cc7c513c9d44ef32843e1fda7ff118cf7c9445...,77867,12507,False,1000,42,NaN
1,F2,2026-11-14T12:02:47,a6fd373f862c7c6a68d92626992dd6bdf08f00b8a26a11...,88097,6351,True,1000,42,"C,E"


production_eligible = False


### Insight

A procedência mostra que os resultados foram carregados de execuções gravadas, sem treino nesta execução: o F1 foi gerado em 2026-11-14 10:53 e o F2 em 2026-11-14 12:02. O ledger tem uma linha para o F1, com `post_hoc` falso (77.867 pedidos de treino e 12.507 de teste), e uma para o F2, com `post_hoc` verdadeiro (88.097 pedidos de treino e 6.351 de teste) e com C e E como variantes aprovadas no gate. Ambos usaram 1.000 reamostragens e a semente 42, e `production_eligible` permanece falso.

As duas linhas registram a consulta ao período de teste, e não a qualidade dos modelos: julho e agosto de 2018 já foram avaliados e estão esgotados para novas avaliações confirmatórias.

## 2. Fonte e população

Os dados são as tabelas do conjunto Olist, lidas de `data/raw/olist/csv/` sem alteração. A população supervisionada é formada pelos pedidos com status `delivered` e com datas de entrega real e estimada válidas, de modo que o alvo possa ser calculado (P-02 em `docs/REGISTRO-DE-DECISOES.md`). Pedidos cancelados, indisponíveis ou ainda em trânsito ficam fora do alvo supervisionado e são tratados em análise separada. A célula seguinte lê somente as tabelas necessárias a este relatório e mostra a distribuição de status e o tamanho da população supervisionada.

In [2]:
from ecom_delivery_sla import build_supervised_orders, read_olist_tables

tables = read_olist_tables(
    project_root / "data" / "raw" / "olist" / "csv",
    table_names=("orders", "items", "customers", "sellers", "geolocation"),
)
orders = tables["orders"]
supervised = build_supervised_orders(orders)

status = orders["order_status"].value_counts().rename_axis("order_status").reset_index(name="orders")
status["share"] = status["orders"] / status["orders"].sum()
display(status)

population = pd.DataFrame(
    {
        "step": [LABELS["all_orders"], LABELS["supervised"]],
        "orders": [len(orders), len(supervised)],
    }
)
population["share"] = population["orders"] / population["orders"].iloc[0]
display(population)
print(
    LABELS["period"].format(
        first=supervised["order_purchase_timestamp"].min().date(),
        last=supervised["order_purchase_timestamp"].max().date(),
    )
)

,order_status,orders,share
0,delivered,96478,0.970203
1,shipped,1107,0.011132
2,canceled,625,0.006285
3,unavailable,609,0.006124
4,invoiced,314,0.003158
5,processing,301,0.003027
6,created,5,0.000050
7,approved,2,0.000020


,step,orders,share
0,Pedidos na tabela de pedidos,99441,1.000000
1,"Pedidos supervisionados (entregues, com datas ...",96470,0.970123


Compras de 2016-09-15 a 2018-08-29


### Insight

A tabela de pedidos tem 99.441 pedidos, dos quais 96.470 (97,0%) formam a população supervisionada. Os pedidos com status `delivered` são 96.478 (97,0%), e o filtro de datas válidas retira ainda alguns deles. Entre os demais status, os maiores são `shipped`, com 1.107 pedidos (1,1%), `canceled`, com 625 (0,6%), e `unavailable`, com 609 (0,6%). As compras cobrem o intervalo de 2016-09-15 a 2018-08-29.

A perda é pequena em volume, mas a população é condicionada à entrega observada, e essa condição pode pesar mais nas janelas recentes, em que pedidos em trânsito ainda não têm desfecho. A seção 3 mostra o efeito dessa censura sobre a prevalência do teste.

## 3. Alvo por dia de calendário e censura

A violação do SLA é definida por dia de calendário: um pedido é marcado como violação quando a data de entrega, truncada para o dia, é posterior à data estimada, também truncada. O horário da entrega, portanto, não conta, e uma entrega feita no próprio dia prometido não é atraso. O atraso em dias, usado apenas na regressão descritiva, é a diferença entre as duas datas.

O fim da série impõe censura à direita (Klein e Moeschberger, 2003): pedidos enviados em julho e agosto de 2018 e ainda sem entrega na extração não têm desfecho observado. Conforme o P-02, esses pedidos são excluídos da população supervisionada, e o cenário que os trata como violação serve apenas de limite superior descritivo, sem alimentar nenhuma métrica de modelo. As tabelas e o gráfico a seguir mostram o alvo na população supervisionada, a sensibilidade à censura na janela de teste do F1 e, por mês de compra, o número de pedidos e a prevalência de violações nos períodos de treino e de teste, sem o mês de junho de 2018.

In [3]:
from ecom_delivery_sla_viz.report import plot_monthly_prevalence

target = pd.DataFrame(
    {
        "orders": [len(supervised)],
        "breaches": [int(supervised["sla_breach"].sum())],
        "prevalence": [supervised["sla_breach"].mean()],
    }
)
display(target)
display(f1["censoring"])
display(f1["monthly_prevalence"])
plot_monthly_prevalence(f1["monthly_prevalence"], language=LANG).show()

,orders,breaches,prevalence
0,96470,6534,0.067731


,scenario,supervised_orders,breaches,censored_orders,prevalence
0,exclude,12507,601,107,0.048053
1,include_as_breach,12614,708,107,0.056128


,purchase_month,split,orders,breaches,prevalence
0,2016-09,train,1,1,1.000000
1,2016-10,train,265,2,0.007547
2,2016-12,train,1,0,0.000000
3,2017-01,train,750,22,0.029333
4,2017-02,train,1653,49,0.029643
5,2017-03,train,2546,116,0.045562
6,2017-04,train,2303,151,0.065567
7,2017-05,train,3545,106,0.029901
8,2017-06,train,3135,95,0.030303
9,2017-07,train,3872,108,0.027893


### Insight

A prevalência global de violações é de 6,8% (6.534 de 96.470 pedidos), mas varia muito entre os meses: de 2,8% em 2017-07 a 12,4% em 2017-11, 14,1% em 2018-02 e 19,0% em 2018-03, voltando a 4,5% em 2018-04. Treino e teste não são estáveis: a prevalência é de 7,5% no treino (até 2018-05) e de 3,4% em julho e 6,2% em agosto de 2018, com 4,8% no conjunto do teste. O teste tem 107 pedidos censurados, e o cenário que os trata como violação elevaria a prevalência de 4,8% para 5,6%, de modo que a censura poderia alterar a prevalência do teste, mas não a levaria ao nível do treino.

As pontas da série têm poucos pedidos e prevalências pouco informativas: 2016-09 tem um pedido, com prevalência de 100,0%, e 2016-12 tem um pedido, enquanto 2016-10 tem 265 pedidos e 0,8%. Variações tão grandes entre meses são compatíveis com mudanças de regime e tornam a calibração sensível ao período de treino, mas a tabela não identifica a causa.

## 4. Features disponíveis na compra e rede as-of

Cada variável do modelo precisa estar disponível no instante da compra. O catálogo de features registra, para cada coluna, a fonte e a disponibilidade temporal, e somente as colunas classificadas como disponíveis na compra e marcadas para uso entram na matriz do modelo. Datas de aprovação, de envio e de entrega ficam fora, porque não existem no momento da compra. As features de rede são calculadas as-of: para cada pedido, as taxas e as medianas históricas do vendedor, da aresta vendedor→UF e da UF de destino usam apenas pedidos já entregues antes da compra, com suavização das taxas, o que preserva a ordem temporal. A célula seguinte lista as features do modelo, agrupadas por fonte e disponibilidade.

In [4]:
from ecom_delivery_sla import feature_catalog

catalog = feature_catalog()
in_model = catalog[catalog["use_in_model"]]
display(in_model.groupby(["source", "status"]).size().rename("features").reset_index())
display(in_model[["feature", "source", "feature_type", "definition"]])

,source,status,features
0,calendar,available_at_purchase,2
1,customer,available_at_purchase,2
2,geolocation,available_at_purchase,2
3,network,available_at_purchase,14
4,order,available_at_purchase,4
5,order_items,available_at_purchase,7
6,payments,available_at_purchase,3
7,products,available_at_purchase,4
8,sellers,available_at_purchase,2


,feature,source,feature_type,definition
5,promised_lead_days,order,numeric,Estimated promise minus purchase
6,customer_state,customer,categorical,Customer destination state
7,customer_is_capital,customer,numeric,Customer city matches the state capital
8,item_count,order_items,numeric,Number of item rows
9,distinct_product_count,order_items,numeric,Number of distinct products
10,distinct_category_count,products,numeric,Number of distinct categories
11,seller_count,order_items,numeric,Number of sellers in the order
12,origin_state_count,sellers,numeric,Number of seller-origin states
13,seller_state,sellers,categorical,State of the seller with highest item value
14,total_price,order_items,numeric,Sum of item prices


### Insight

A matriz do modelo reúne features de fontes distintas, todas classificadas como `available_at_purchase`. A rede as-of é a maior fonte, com 14 features, seguida de `order_items` (7), `order` (4), `products` (4) e `payments` (3), e as demais fontes têm 2 features cada. A distância entre vendedor e cliente entra como `distance_mean_km`, a média por pedido adotada em P-03, acompanhada de `same_state_share`. O prazo prometido entra como `promised_lead_days`, da fonte `order`, e `promise_slack`, da rede, compara esse prazo com a mediana histórica de entrega.

A disponibilidade temporal não torna uma feature causal. As features de rede dependem de histórico anterior, e as colunas `seller_cold_start` e `edge_cold_start` sinalizam pedidos de vendedores e rotas ainda sem histórico, para os quais as taxas suavizadas carregam pouca informação.

O mapa a seguir é descritivo e agrega todo o período: cada seta representa uma rota entre UFs, com a unidade par pedido-vendedor, espessura proporcional ao volume e cor dada pela taxa de atraso. O intervalo de confiança da taxa de cada rota usa o método de Wilson (Wilson, 1927), visível no *hover*, e o menu do gráfico alterna o volume mínimo de pares por rota. Como o mapa usa todos os desfechos do período, ele não é uma feature nem alimenta o modelo; mostra a estrutura da rede da qual as features as-of são extraídas. A tabela mostra as maiores rotas entre UFs.

In [5]:
from ecom_delivery_sla_viz import plot_route_network, route_pairs, route_table

route_pair_table = route_pairs(
    supervised,
    tables["items"],
    tables["sellers"],
    tables["customers"],
    tables["geolocation"],
)
display(route_table(route_pair_table, level="state", min_orders=100).head(10))
plot_route_network(route_pair_table, tables["geolocation"], language=LANG).show()

,origin,destination,origin_state,destination_state,orders,breaches,breach_rate,ci_low,ci_high,share,distance_median_km,promised_median_days,actual_median_days,is_self_loop
0,SP,SP,SP,SP,31180,1431,0.045895,0.043628,0.048274,0.318778,94.967364,17.150017,6.518657,True
1,SP,RJ,SP,RJ,8274,1157,0.139836,0.132530,0.147476,0.084592,383.754544,24.421858,12.766146,False
2,SP,MG,SP,MG,7562,394,0.052103,0.047317,0.057343,0.077312,488.720494,23.434583,10.770683,False
3,SP,RS,SP,RS,3662,238,0.064992,0.057455,0.073441,0.037440,867.538356,28.673304,13.827477,False
4,SP,PR,SP,PR,3164,138,0.043616,0.037035,0.051303,0.032348,419.257758,24.293831,11.066348,False
5,PR,SP,PR,SP,2975,90,0.030252,0.024677,0.037038,0.030416,469.147835,24.617072,9.431586,False
6,MG,SP,MG,SP,2581,91,0.035258,0.028805,0.043092,0.026388,467.009680,22.458275,9.195069,False
7,SP,SC,SP,SC,2355,222,0.094268,0.083118,0.106739,0.024077,544.157927,25.279525,13.831308,False
8,SP,BA,SP,BA,2340,297,0.126923,0.114044,0.141025,0.023924,1448.330267,29.301134,17.584491,False
9,MG,MG,MG,MG,1542,39,0.025292,0.018556,0.034387,0.015765,239.548351,19.402141,7.263767,True


### Insight

O volume é concentrado: a rota SP→SP responde por 31.180 pares (31,9%), e a origem SP domina as rotas de maior volume, como SP→RJ (8.274 pares), SP→MG (7.562) e SP→RS (3.662). A taxa de atraso muda de forma visível entre os destinos: é de 14,0% em SP→RJ (IC de 13,3% a 14,7%) e de 12,7% em SP→BA, contra 4,6% em SP→SP (IC de 4,4% a 4,8%) e 5,2% em SP→MG. As rotas de maior taxa também têm prazo prometido mais longo, com mediana de 24,4 dias em SP→RJ contra 17,2 dias em SP→SP.

Rotas de volume pequeno têm intervalos largos e não sustentam comparação isolada, e a tabela exibida traz apenas as rotas de maior volume. O mapa é descritivo: a associação entre rota e atraso não indica que a rota cause o atraso, pois distância, composição de produtos e vendedores variam junto com ela.

## 5. Protocolo temporal e baselines obrigatórias

Nenhum modelo é avaliado fora da ordem do tempo. O desenvolvimento usa folds expanding-window mensais (Bergmeir e Benítez, 2012): o treino acumula todo o passado, um embargo de 30 dias separa treino e validação, e cada fold valida um mês, de outubro de 2017 a junho de 2018. O pré-processamento é ajustado somente no treino de cada fold, e os estimadores usam a implementação do scikit-learn (Pedregosa et al., 2011) e das bibliotecas de boosting. Nenhum modelo usa pesos de classe, e os classificadores são calibrados com regressão isotônica (Zadrozny e Elkan, 2002; Niculescu-Mizil e Caruana, 2005) ajustada apenas no treino.

A métrica principal da classificação é a precisão média, ou average precision (AP) (Davis e Goadrich, 2006), adequada a uma classe minoritária como a violação do prazo. O escore de Brier (Brier, 1950) e o erro de calibração esperado em dez faixas, ECE (Naeini et al., 2015), avaliam as probabilidades. A métrica principal da regressão é o MAE. Toda comparação inclui as baselines obrigatórias: a média do treino e as medianas do atraso, a baseline por UF, a prevalência, a regra do prazo prometido negativo e a regressão logística com prazo prometido e UF. Um modelo só é candidato se superar a melhor baseline pelo critério da seção seguinte. As tabelas a seguir mostram a configuração gravada do torneio, os modelos e as baselines de cada tarefa e as janelas de treino e de teste do F1.

In [6]:
classification_results = tournament["classification_results"]
regression_results = tournament["regression_results"]
config = tournament_meta["config"]

protocol = pd.DataFrame(
    {
        "item": [
            LABELS["protocol"]["first_val_month"],
            LABELS["protocol"]["last_val_month"],
            LABELS["protocol"]["embargo_days"],
            LABELS["protocol"]["random_state"],
            LABELS["protocol"]["n_folds"],
        ],
        "value": [
            config["first_val_month"],
            config["last_val_month"],
            config["embargo_days"],
            config["random_state"],
            classification_results["fold"].nunique(),
        ],
    }
)
display(protocol)

roles = []
for task, results in (("classification", classification_results), ("regression", regression_results)):
    part = results.groupby("model", sort=False)["fold"].nunique().rename("folds").reset_index()
    part.insert(0, "task", LABELS["task"][task])
    part["role"] = part["model"].str.startswith("baseline_").map({True: LABELS["baseline"], False: LABELS["model"]})
    roles.append(part)
display(pd.concat(roles, ignore_index=True))
display(f1["populations"])

,item,value
0,Primeiro mês de validação,2017-10
1,Último mês de validação,2018-06
2,Embargo entre treino e validação (dias),30
3,Semente,42
4,Folds de validação,9


,task,model,folds,role
0,classificação,LogisticRegression,9,modelo candidato
1,classificação,RandomForest,9,modelo candidato
2,classificação,ExtraTrees,9,modelo candidato
3,classificação,HistGradientBoosting,9,modelo candidato
4,classificação,XGBoost,9,modelo candidato
5,classificação,LightGBM,9,modelo candidato
6,classificação,CatBoost,9,modelo candidato
7,classificação,baseline_prevalence,9,baseline obrigatória
8,classificação,baseline_negative_promised_lead_days,9,baseline obrigatória
9,classificação,baseline_logistic_promise_plus_state,9,baseline obrigatória


,split,window_start,window_end_exclusive,orders,breaches,prevalence,first_purchase,last_purchase,weeks
0,train,None,2018-06-01,77867,5862,0.075282,2016-09-15 12:16:38,2018-05-31 23:51:24,NaN
1,test,2018-07-01,2018-09-01,12507,601,0.048053,2018-07-01 00:25:07,2018-08-29 15:00:37,10.0


### Insight

O protocolo corresponde ao descrito: 9 folds mensais de validação, de 2017-10 a 2018-06, com embargo de 30 dias e semente 42. As baselines obrigatórias estão presentes nas duas tarefas: prevalência, prazo prometido negativo e regressão logística com prazo e UF na classificação, e média do treino, mediana do atraso e mediana por UF na regressão. O treino do F1 termina antes de 2018-06-01, com 77.867 pedidos e prevalência de 7,5%, e o teste cobre de 2018-07-01 a 2018-09-01 (exclusivo), com 12.507 pedidos e prevalência de 4,8%, de modo que junho de 2018 fica fora das duas janelas.

O teste dispõe de apenas 10 semanas, o que limita o número de blocos do bootstrap e alarga os intervalos. A prevalência do teste (4,8%) também difere da do treino (7,5%), e essa diferença condiciona a leitura das métricas de calibração adiante.

## 6. Resultado do desenvolvimento

O critério de seleção foi aprovado antes da execução (P-01). Um modelo torna-se candidato quando o ganho relativo médio sobre a melhor baseline atinge ao menos 5% em AP, na classificação, ou 3% em MAE, na regressão, e o modelo vence essa baseline em pelo menos 75% dos folds, o que equivale a sete dos nove folds do desenvolvimento. Para a classificação, o ganho relativo é

$$G_m = \frac{\overline{AP}_m - \overline{AP}_b}{\overline{AP}_b}$$

Onde:

- $G_m$: ganho relativo médio do modelo $m$ sobre a melhor baseline;
- $\overline{AP}_m$: média da AP do modelo $m$ nos folds de validação;
- $\overline{AP}_b$: média da AP da melhor baseline $b$, definida como a baseline de maior AP média nos mesmos folds.

O ganho é positivo quando o modelo supera a melhor baseline na média dos folds, e a contagem de folds vencidos mede a consistência dessa vantagem. Entre os modelos comparados está o CatBoost (Prokhorenkova et al., 2018). A tabela traz as métricas médias e o número de folds vencidos. O primeiro gráfico mostra a AP de cada modelo em cada fold, com a melhor baseline tracejada, e o segundo mostra o ganho médio de cada modelo com a linha do critério.

In [7]:
from ecom_delivery_sla import tournament_summary
from ecom_delivery_sla_viz import plot_gain_over_baseline, plot_metric_by_fold

classification_summary = tournament_summary(classification_results)
summary_columns = [
    "model",
    "n_folds",
    "average_precision_mean",
    "average_precision_std",
    "average_precision_gain_vs_baseline",
    "average_precision_folds_won",
    "average_precision_best_baseline",
    "ece_10bins_mean",
    "fit_seconds_mean",
]
display(classification_summary[[c for c in summary_columns if c in classification_summary.columns]])

candidates = classification_summary.loc[~classification_summary["is_baseline"], "model"].tolist()
best_baseline = classification_summary["average_precision_best_baseline"].replace("", pd.NA).dropna().iloc[0]
plot_metric_by_fold(
    classification_results,
    fold_col="fold",
    model_col="model",
    metric_col="average_precision",
    reference=best_baseline,
    models=candidates,
    language=LANG,
).show()
plot_gain_over_baseline(classification_summary, metric="average_precision", threshold=0.05, fold_results=classification_results, language=LANG).show()

,model,n_folds,average_precision_mean,average_precision_std,average_precision_gain_vs_baseline,average_precision_folds_won,average_precision_best_baseline,ece_10bins_mean,fit_seconds_mean
0,CatBoost,9,0.174999,0.101000,0.139115,8.0,baseline_logistic_promise_plus_state,0.057619,23.386609
1,HistGradientBoosting,9,0.156258,0.098451,0.017128,7.0,baseline_logistic_promise_plus_state,0.064149,10.748787
2,XGBoost,9,0.154524,0.086892,0.005840,6.0,baseline_logistic_promise_plus_state,0.059344,11.419012
3,baseline_logistic_promise_plus_state,9,0.153627,0.089007,NaN,NaN,,0.049702,NaN
4,LightGBM,9,0.152469,0.094095,-0.007537,4.0,baseline_logistic_promise_plus_state,0.062677,5.082216
5,LogisticRegression,9,0.151856,0.089213,-0.011531,4.0,baseline_logistic_promise_plus_state,0.079425,7.809579
6,RandomForest,9,0.150238,0.097009,-0.022061,3.0,baseline_logistic_promise_plus_state,0.057769,55.814494
7,ExtraTrees,9,0.149607,0.095750,-0.026169,5.0,baseline_logistic_promise_plus_state,0.057258,46.056691
8,baseline_prevalence,9,0.083395,0.056714,NaN,NaN,,0.050500,NaN
9,baseline_negative_promised_lead_days,9,0.078068,0.046667,NaN,NaN,,NaN,NaN


### Insight

Apenas o CatBoost atinge o critério pré-registrado: AP média de 0,175 contra 0,154 da melhor baseline (regressão logística com prazo e UF), ganho relativo de 13,9% e vitória em 8 de 9 folds, o que satisfaz as duas condições do critério. O HistGradientBoosting vence em 7 de 9 folds, mas com ganho de apenas 1,7%, e o XGBoost tem ganho de 0,6% em 6 folds. O LightGBM (−0,8%) e a regressão logística (−1,2%) ficam abaixo da baseline, e as baselines de prevalência (0,083) e de prazo prometido negativo (0,078) ficam bem abaixo dos modelos.

A vantagem é de magnitude irregular entre os meses: o desvio-padrão da AP do CatBoost entre folds é 0,101, e a AP da configuração do F1 varia de 0,070 em 2018-06 a 0,350 em 2018-03, mês de maior prevalência (19,0%). Na calibração, o CatBoost (ECE de 0,058) não supera a baseline logística (0,050), e seu ajuste leva 23,4 s em média. O ganho é relativo a uma baseline de AP baixa, e a vantagem no desenvolvimento não implica desempenho em dados posteriores.

Na regressão do atraso, o mesmo critério se aplica ao MAE, em que menor é melhor, com ganho mínimo de 3%. A baseline de referência é escolhida, como na classificação, pela melhor média nos folds. O primeiro gráfico mostra o MAE de cada modelo em cada fold, com a melhor baseline tracejada, e o segundo mostra o ganho médio com a linha do critério.

In [8]:
regression_summary = tournament_summary(regression_results)
regression_columns = [
    "model",
    "n_folds",
    "mae_mean",
    "mae_std",
    "mae_gain_vs_baseline",
    "mae_folds_won",
    "mae_best_baseline",
    "rmse_mean",
]
display(regression_summary[[c for c in regression_columns if c in regression_summary.columns]])

regression_candidates = regression_summary.loc[~regression_summary["is_baseline"], "model"].tolist()
regression_baseline = regression_summary["mae_best_baseline"].replace("", pd.NA).dropna().iloc[0]
plot_metric_by_fold(
    regression_results,
    fold_col="fold",
    model_col="model",
    metric_col="mae",
    reference=regression_baseline,
    models=regression_candidates,
    lower_is_better=True,
    language=LANG,
).show()
plot_gain_over_baseline(regression_summary, metric="mae", threshold=0.03, fold_results=regression_results, language=LANG).show()

,model,n_folds,mae_mean,mae_std,mae_gain_vs_baseline,mae_folds_won,mae_best_baseline,rmse_mean
0,ExtraTrees,9,5.953134,0.846722,-0.002634,5.0,baseline_state_median_actual_minus_promise,9.086746
1,CatBoost,9,5.947470,1.085944,-0.001680,5.0,baseline_state_median_actual_minus_promise,9.091165
2,HistGradientBoosting,9,6.056313,1.498375,-0.020011,6.0,baseline_state_median_actual_minus_promise,9.167484
3,baseline_state_median_actual_minus_promise,9,5.937495,1.395744,NaN,NaN,,9.203186
4,ElasticNet,9,6.300179,1.085398,-0.061084,4.0,baseline_state_median_actual_minus_promise,9.283756
5,RandomForest,9,6.380546,1.783798,-0.074619,6.0,baseline_state_median_actual_minus_promise,9.298089
6,XGBoost,9,6.330275,1.453125,-0.066152,4.0,baseline_state_median_actual_minus_promise,9.352481
7,LightGBM,9,6.331933,1.447127,-0.066432,4.0,baseline_state_median_actual_minus_promise,9.414514
8,Ridge,9,6.841355,1.380036,-0.152229,2.0,baseline_state_median_actual_minus_promise,9.833554
9,OLS,9,6.862683,1.386298,-0.155821,2.0,baseline_state_median_actual_minus_promise,9.855199


### Insight

Nenhum modelo de regressão atinge o critério de ganho mínimo em MAE. O ExtraTrees (MAE de 5,953) e o CatBoost (5,947) ficam ligeiramente acima da baseline por UF (5,937), com ganhos de −0,3% e −0,2% e vitória em 5 de 9 folds cada. O HistGradientBoosting vence em 6 folds, mas com ganho de −2,0%, e os modelos lineares ficam piores (Ridge −15,2% e OLS −15,6%). Esse resultado é coerente com o registro de P-04: a regressão não teve candidato.

As diferenças entre os melhores modelos e a baseline aparecem na terceira casa decimal e são muito menores que o desvio-padrão do MAE entre folds (0,847 no ExtraTrees e 1,396 na baseline por UF). O RMSE segue um padrão parecido, com 9,087 no ExtraTrees e 9,091 no CatBoost contra 9,203 da baseline. O atraso em dias permanece como análise descritiva, com a baseline por UF como referência.

A ablação da feature de distância testou se uma medida diferente da distância média entre vendedor e cliente, a distância máxima ou a máxima acrescida da amplitude, melhoraria o modelo em pedidos com mais de um vendedor. A regra de adoção foi fixada antes da execução (P-03): uma variante substituiria a distância média apenas se melhorasse a métrica principal na média dos folds e vencesse a referência em pelo menos sete dos nove folds. As tabelas trazem as médias e os folds vencidos na validação completa e no subgrupo multi-vendedor, e os gráficos mostram a métrica principal por variante, com o desvio entre folds.

In [9]:
from ecom_delivery_sla import ablation_summary
from ecom_delivery_sla_viz import plot_ablation

for task in ("classification", "regression"):
    task_summary = ablation_summary(ablation[f"{task}_ablation"], reference="mean", task=task)
    display(task_summary)
    plot_ablation(task_summary, language=LANG).show()

,variant,model,n_folds,multi_seller_n_mean,multi_seller_n_folds,average_precision_mean,average_precision_std,average_precision_delta_vs_mean,average_precision_folds_won,multi_seller_average_precision_mean,multi_seller_average_precision_std,multi_seller_average_precision_delta_vs_mean,multi_seller_average_precision_folds_won
0,mean,LogisticRegression,9,89.0,5,0.151856,0.089213,NaN,NaN,0.149058,0.198157,NaN,NaN
1,mean,HistGradientBoosting,9,89.0,5,0.156258,0.098451,NaN,NaN,0.155513,0.096495,NaN,NaN
2,max,LogisticRegression,9,89.0,5,0.152102,0.089103,0.001626,5.0,0.148604,0.198277,-0.003044,1.0
3,max,HistGradientBoosting,9,89.0,5,0.157362,0.094520,0.007064,5.0,0.211612,0.183561,0.360729,2.0
4,max_plus_range,LogisticRegression,9,89.0,5,0.151863,0.089530,0.000050,5.0,0.164253,0.195013,0.101939,2.0
5,max_plus_range,HistGradientBoosting,9,89.0,5,0.157844,0.095896,0.010147,6.0,0.107461,0.058194,-0.308992,1.0


,variant,model,n_folds,multi_seller_n_mean,multi_seller_n_folds,mae_mean,mae_std,mae_delta_vs_mean,mae_folds_won,multi_seller_mae_mean,multi_seller_mae_std,multi_seller_mae_delta_vs_mean,multi_seller_mae_folds_won
0,mean,ElasticNet,9,89.0,9,6.300179,1.085398,NaN,NaN,4.664334,1.118722,NaN,NaN
1,mean,HistGradientBoosting,9,89.0,9,6.056313,1.498375,NaN,NaN,4.464082,1.853946,NaN,NaN
2,max,ElasticNet,9,89.0,9,6.300774,1.085398,-0.000094,1.0,4.684209,1.116501,-0.004261,2.0
3,max,HistGradientBoosting,9,89.0,9,6.055120,1.476146,0.000197,5.0,4.517004,1.921493,-0.011855,2.0
4,max_plus_range,ElasticNet,9,89.0,9,6.299801,1.085478,0.000060,6.0,4.650839,1.121373,0.002893,5.0
5,max_plus_range,HistGradientBoosting,9,89.0,9,6.063650,1.493304,-0.001212,3.0,4.512154,1.873316,-0.010769,4.0


### Insight

Nenhuma variante cumpre a regra de adoção. Na classificação com HistGradientBoosting, a AP média passa de 0,156 (distância média) para 0,157 (máxima) e 0,158 (máxima mais amplitude), ganhos relativos de 0,7% e 1,0%, com vitória em 5 e 6 folds de 9, abaixo do mínimo exigido. Na regressão logística, a AP é de 0,152 nas três versões, com ganhos de 0,2% e 0,0%, e na regressão do atraso os ganhos relativos de MAE ficam entre −0,1% e 0,0%. Essas diferenças são muito menores que o desvio-padrão da AP entre folds (0,089 a 0,098), e a distância média foi mantida, como registrado em P-03.

O subgrupo multi-vendedor tem em média 89 pedidos por fold e aparece em 5 folds na classificação. Sua AP oscila sem coerência: no HistGradientBoosting, +36,1% com a máxima e −30,9% com a máxima mais amplitude, e na regressão logística, −0,3% e +10,2%, com desvio-padrão entre folds de 0,058 a 0,198. Isso é compatível com ruído de amostra pequena, sem indício de efeito, e o subgrupo foi reportado à parte, sem decidir a escolha.

Os estudos do Optuna (Akiba et al., 2019) da versão `V4_folds` exploraram em conjunto o pré-processamento e os hiperparâmetros de cada família de modelos, com o mesmo protocolo de folds. Eles verificam se um ajuste fino alteraria a leitura do torneio; a configuração do F1, fixada antes, permanece. A busca da classificação otimiza a AP, a mesma métrica do critério P-01, e a da regressão otimiza o RMSE, métrica diferente da usada no critério de ganho em MAE. A tabela traz o melhor valor de cada estudo e sua posição dentro da tarefa, sem declarar vencedor. Os estudos são apenas lidos do banco SQLite, e nenhuma tentativa nova é executada.

In [10]:
import optuna

from ecom_delivery_sla_viz.report import optuna_study_table

optuna.logging.set_verbosity(optuna.logging.WARNING)
summaries = optuna.get_all_study_summaries(storage=f"sqlite:///{OPTUNA_DB.as_posix()}")
display(optuna_study_table(summaries, version=STUDY_VERSION))

,task,model,trials,metric,direction,best_value,rank
0,classification,LightGBM,100,validation_average_precision,maximize,0.174243,1.0
1,classification,CatBoost,100,validation_average_precision,maximize,0.172464,2.0
2,classification,XGBoost,100,validation_average_precision,maximize,0.171975,3.0
3,classification,LogisticRegression,100,validation_average_precision,maximize,0.165821,4.0
4,regression,CatBoost,100,validation_rmse,minimize,8.846497,1.0
5,regression,XGBoost,100,validation_rmse,minimize,9.054772,2.0
6,regression,LightGBM,101,validation_rmse,minimize,9.098937,3.0
7,regression,ElasticNet,100,validation_rmse,minimize,9.121467,4.0
8,regression,Ridge,100,validation_rmse,minimize,9.157278,5.0


### Insight

Na classificação, o melhor valor de cada estudo é 0,174 (LightGBM), 0,172 (CatBoost), 0,172 (XGBoost) e 0,166 (regressão logística), com 100 tentativas por família. A ordem mudou em relação ao torneio, em que o CatBoost com parâmetros padrão (0,175) liderava e o LightGBM tinha 0,152: o ajuste elevou o LightGBM e a regressão logística (de 0,152 para 0,166), mas nenhum estudo superou o CatBoost padrão. As diferenças entre os três primeiros estudos são da ordem da terceira casa decimal, bem menores que o desvio-padrão entre folds do torneio (0,094 a 0,101), e não bastam para declarar um vencedor.

Os valores dos estudos são AP de validação obtida no protocolo de folds usado na busca, o que é compatível com otimismo, e a otimização não altera o critério de candidato nem a configuração pré-registrada do F1. Na regressão, o melhor RMSE é 8,846 (CatBoost), contra 9,091 do CatBoost padrão no torneio e 9,203 da baseline por UF, mas o critério usa o MAE, e por isso essa comparação não reabre a decisão de que a regressão não teve candidato.

## 7. F1: resultado oficial (inconclusivo)

O F1 abriu o período de teste uma única vez, conforme o pré-registro P-05. O modelo é o CatBoost com a configuração padrão do torneio, sem pesos de classe e com calibração isotônica ajustada no treino, e as mesmas features do desenvolvimento. O treino usa as compras anteriores a 2018-06-01, e o mês de junho de 2018 fica fora de todas as etapas. O teste cobre as compras de 2018-07-01 a 2018-08-31, com os pedidos censurados excluídos. As baselines são ajustadas na mesma janela de treino, e a referência principal é a regressão logística com prazo prometido e UF. O resultado é congelado: a versão gravada é sempre carregada, nunca recalculada, e cada cálculo acrescenta uma linha ao ledger.

A leitura é decidida antes do resultado. A diferença pareada de AP entre o CatBoost e a referência principal é

$$\Delta = AP_{\text{CatBoost}} - AP_{\text{log}}$$

Onde:

- $\Delta$: diferença pareada de AP, avaliada nas mesmas reamostragens do bootstrap;
- $AP_{\text{CatBoost}}$: AP do CatBoost no período de teste;
- $AP_{\text{log}}$: AP da regressão logística com prazo prometido e UF no mesmo período.

O resultado é confirmado se $\Delta$ for positiva e o limite inferior do IC 95% for maior que zero, não confirmado se $\Delta$ for negativa e o limite superior for menor que zero, e inconclusivo nos demais casos. O IC vem de bootstrap por blocos semanais da semana ISO da compra (Künsch, 1989), com 1.000 reamostragens e semente 42. As tabelas a seguir mostram a procedência do resultado e as populações de treino e de teste.

In [11]:
display(f1["provenance"])
display(f1["populations"])

,version,post_hoc,created_at,fingerprint,train_orders,test_orders,train_end_exclusive,test_start,test_end_exclusive,n_boot,seed,model_path
0,F1,False,2026-11-14T10:53:16,ddc0ef97cc7c513c9d44ef32843e1fda7ff118cf7c9445...,77867,12507,2018-06-01,2018-07-01,2018-09-01,1000,42,d:\Projetos\E-commerce-Delivery-Network-&-SLA-...


,split,window_start,window_end_exclusive,orders,breaches,prevalence,first_purchase,last_purchase,weeks
0,train,None,2018-06-01,77867,5862,0.075282,2016-09-15 12:16:38,2018-05-31 23:51:24,NaN
1,test,2018-07-01,2018-09-01,12507,601,0.048053,2018-07-01 00:25:07,2018-08-29 15:00:37,10.0


### Insight

A procedência registra uma única linha para o F1, com `post_hoc` falso, criada em 2026-11-14 10:53 com 1.000 reamostragens e semente 42, de modo que o resultado foi calculado uma única vez e depois apenas carregado. O treino tem 77.867 pedidos e 5.862 violações (7,5%), e o teste tem 12.507 pedidos e 601 violações (4,8%), em 10 semanas disponíveis para o bootstrap por blocos semanais.

Com apenas 10 blocos, os intervalos tendem a ser largos, e a prevalência menor no teste deixa menos violações (601) para estimar a AP. A diferença entre as prevalências de treino e de teste é compatível com mudança de regime entre os períodos, mas essa tabela não permite explicá-la.

A primeira tabela traz, para o CatBoost e para as baselines, a estimativa pontual de cada métrica na amostra completa e o IC 95% do bootstrap, com todos os modelos avaliados nas mesmas reamostragens. A segunda traz a diferença pareada de AP em relação à referência principal e a leitura segue a regra pré-registrada. O gráfico reúne a AP de cada modelo com seu intervalo e, ao lado, a diferença pareada em relação à linha de zero.

In [12]:
from ecom_delivery_sla_viz import plot_metric_intervals

metrics_f1 = f1["classification_metrics"]
display(metrics_f1[["model", "metric", "estimate", "ci_low", "ci_high"]])
print(
    LABELS["discarded"].format(
        n=int(metrics_f1["n_discarded"].iloc[0]),
        total=int(metrics_f1["n_boot"].iloc[0]),
    )
)

paired_f1 = f1["paired_difference"]
display(paired_f1[["model", "reference", "estimate_model", "estimate_reference", "difference", "ci_low", "ci_high"]])
outcome = paired_f1.loc[0, "outcome"] if LANG == "pt" else paired_f1.loc[0, "outcome_en"]
print(LABELS["reading"].format(outcome=outcome))
plot_metric_intervals(metrics_f1, metric="average_precision", paired=paired_f1, language=LANG).show()

,model,metric,estimate,ci_low,ci_high
0,CatBoost,average_precision,0.081983,0.055835,0.133698
1,CatBoost,roc_auc,0.610157,0.561247,0.661876
2,CatBoost,brier,0.050476,0.031599,0.068670
3,CatBoost,ece_10bins,0.038628,0.033804,0.054392
4,CatBoost,precision_at_5pct,0.108626,0.079020,0.152373
5,CatBoost,recall_at_5pct,0.113145,0.070758,0.225214
6,CatBoost,precision_at_10pct,0.084732,0.058771,0.110333
7,CatBoost,recall_at_10pct,0.176373,0.120073,0.305240
8,baseline_prevalence,average_precision,0.048053,0.027216,0.068543
9,baseline_prevalence,roc_auc,0.500000,0.500000,0.500000


Reamostragens descartadas por conter uma só classe: 0 de 1000


,model,reference,estimate_model,estimate_reference,difference,ci_low,ci_high
0,CatBoost,baseline_logistic_promise_plus_state,0.081983,0.08425,-0.002267,-0.032492,0.063988


Leitura pré-registrada da diferença pareada de AP (CatBoost - baseline logística): inconclusivo


### Insight

A leitura pré-registrada da diferença pareada é “inconclusivo”. A AP do CatBoost é 0,082 (IC 95% de 0,056 a 0,134) e a da regressão logística com prazo e UF é 0,084 (0,056 a 0,116), e a diferença pareada é −0,002, com IC de −0,032 a +0,064, que inclui zero. Os intervalos dos dois modelos se sobrepõem quase por completo, e os do CatBoost e da baseline de prevalência (0,048; 0,027 a 0,069) também se sobrepõem. A regra do prazo prometido negativo tem a maior AP pontual (0,142; IC de 0,044 a 0,232) e AUC de 0,725, contra 0,610 do CatBoost, mas seu intervalo largo engloba os demais, e a comparação pareada oficial é apenas com a baseline logística.

O ganho de 13,9% do desenvolvimento não se reproduziu como confirmação no teste. Intervalos largos refletem o número limitado de semanas, e a diferença pareada não equivale a evidência de ausência de efeito: o IC é compatível tanto com uma perda de 0,032 quanto com um ganho de 0,064 de AP.

A curva de confiabilidade (DeGroot e Fienberg, 1983) compara, em dez faixas de probabilidade, a probabilidade média prevista com a taxa observada de violações, para os modelos que produzem probabilidades. Pontos sobre a diagonal indicam calibração. A tabela de precisão e recall nos 5% e 10% de maior risco resume o uso do escore para priorização.

In [13]:
from ecom_delivery_sla_viz.report import plot_reliability_curves

reliability_f1 = f1["reliability"]
display(reliability_f1)
plot_reliability_curves(reliability_f1, language=LANG).show()
display(f1["top_k"])

,model,bin,bin_lower,bin_upper,n,mean_predicted,observed_rate
0,CatBoost,1,0.0,0.1,9080,0.048305,0.043833
1,CatBoost,2,0.1,0.2,2227,0.140257,0.044903
2,CatBoost,3,0.2,0.3,881,0.243339,0.063564
3,CatBoost,4,0.3,0.4,244,0.338651,0.106557
4,CatBoost,5,0.4,0.5,57,0.437436,0.245614
5,CatBoost,6,0.5,0.6,9,0.548315,0.555556
6,CatBoost,7,0.6,0.7,5,0.632761,0.200000
7,CatBoost,8,0.7,0.8,3,0.728085,0.333333
8,CatBoost,9,0.8,0.9,1,0.800651,0.000000
9,CatBoost,10,0.9,1.0,0,NaN,NaN


,model,top_fraction,precision,precision_ci_low,precision_ci_high,recall,recall_ci_low,recall_ci_high
0,CatBoost,0.05,0.108626,0.079020,0.152373,0.113145,0.070758,0.225214
1,CatBoost,0.10,0.084732,0.058771,0.110333,0.176373,0.120073,0.305240
2,baseline_prevalence,0.05,0.047923,0.009601,0.093909,0.049917,0.010360,0.100013
3,baseline_prevalence,0.10,0.045564,0.013006,0.096087,0.094842,0.027850,0.217651
4,baseline_negative_promised_lead_days,0.05,0.190096,0.064200,0.283750,0.198003,0.105552,0.257891
5,baseline_negative_promised_lead_days,0.10,0.167866,0.058809,0.239437,0.349418,0.195506,0.418261
6,baseline_logistic_promise_plus_state,0.05,0.115016,0.078062,0.160643,0.119800,0.096207,0.154398
7,baseline_logistic_promise_plus_state,0.10,0.082334,0.056623,0.108987,0.171381,0.140491,0.226266


### Insight

O CatBoost superestima o risco nas faixas intermediárias: de 0,1 a 0,2, a probabilidade média prevista é 0,140 e a taxa observada 0,045 (2.227 pedidos); de 0,2 a 0,3, 0,243 contra 0,064 (881 pedidos); de 0,3 a 0,4, 0,339 contra 0,107 (244 pedidos). A faixa de 0 a 0,1 concentra 9.080 pedidos, com 0,048 previsto e 0,044 observado, e as faixas acima de 0,5 têm no máximo 9 pedidos, de modo que suas taxas são instáveis. A regressão logística tem direção semelhante (0,134 previsto contra 0,059 observado de 0,1 a 0,2). O ECE do CatBoost é 0,039, menor que o da logística (0,054), porque a maior parte dos pedidos está na faixa mais baixa, e por isso o ECE resume pouco a distorção nas faixas altas.

A calibração isotônica foi ajustada somente no treino, com prevalência de 7,5%, e o teste tem 4,8%, de modo que o desvio é compatível com a mudança de distribuição, o que não foi testado. Na priorização, a precisão do CatBoost nos 5% de maior risco é 0,109 (IC de 0,079 a 0,152), próxima da logística (0,115) e abaixo da regra do prazo prometido negativo (0,190; IC de 0,064 a 0,284). Nos 10% de maior risco, os valores são 0,085, 0,082 e 0,168, e a baseline de prevalência dá 0,048 e 0,046, o piso de referência.

O PSI, índice de estabilidade populacional (Siddiqi, 2006), compara cada feature numérica entre a janela de treino e a de teste, com os quantis definidos no treino. O gráfico ordena as features em escala logarítmica, com linhas de referência em 0,10 e em 0,25, e as features de calendário aparecem em cinza porque o PSI delas é alto por construção, já que o teste cobre apenas dois meses. A tabela traz as dez features de maior PSI. O PSI é um diagnóstico descritivo do deslocamento marginal e não mede o efeito sobre o modelo.

In [14]:
from ecom_delivery_sla_viz import plot_psi

psi = f1["psi"]
display(psi.head(10))
calendar_features = ["purchase_month", "days_to_black_friday", "days_to_christmas"]
plot_psi(psi, neutral_features=calendar_features, highlight_col="highlight", language=LANG).show()

,feature,psi,highlight
0,days_to_black_friday,10.913776,other
1,days_to_christmas,10.741474,other
2,purchase_month,10.687575,other
3,dest_hist_breach_rate,6.303497,network_rate
4,dest_hist_median_lead_days,4.301667,other
5,dest_in_degree,1.395556,other
6,seller_hist_breach_rate,1.317856,network_rate
7,edge_hist_breach_rate,1.145810,network_rate
8,promise_slack,0.957536,other
9,promised_lead_days,0.767983,promise


### Insight

As features de calendário têm os maiores PSI (`days_to_black_friday` com 10,914, `days_to_christmas` com 10,741 e `purchase_month` com 10,688), como esperado por construção. Fora delas, o maior deslocamento é o de `dest_hist_breach_rate` (6,303), seguido de `dest_hist_median_lead_days` (4,302), `dest_in_degree` (1,396), `seller_hist_breach_rate` (1,318), `edge_hist_breach_rate` (1,146) e `promise_slack` (0,958). O prazo prometido, `promised_lead_days`, tem PSI de 0,768. O PSI de todas essas features é alto, muito acima das linhas de referência do gráfico.

O deslocamento das taxas históricas da rede e do prazo prometido é compatível com a mudança da política de promessa, cuja mediana vai de 21,3 a 27,4 dias entre 2017-03 e 2018-06 para 19,3 dias em julho e 13,4 dias em agosto de 2018, e com o viés de atraso de rótulo no histórico da rede. O PSI descreve o deslocamento marginal de cada feature, não mede o efeito sobre o modelo e não permite atribuição causal.

A regressão do atraso foi avaliada no teste apenas de forma descritiva, porque não tinha candidato pré-registrado (P-04). A tabela traz o MAE e o RMSE em dias do CatBoost e das baselines, ordenados pelo MAE só para facilitar a leitura. A ordenação não é seleção de modelo, e nenhum intervalo de confiança é calculado para a regressão.

In [15]:
display(f1["regression"].sort_values("mae"))

,model,mae,rmse,n_test
3,CatBoost,3.885680,5.331360,12507
2,baseline_state_median_actual_minus_promise,4.514864,5.723027,12507
1,baseline_median_actual_minus_promise,4.927942,6.079787,12507
0,baseline_train_mean,6.392634,8.215881,12507


### Insight

Na regressão do teste, o CatBoost tem MAE de 3,886 dias e RMSE de 5,331, contra 4,515 e 5,723 da baseline por UF, 4,928 e 6,080 da mediana global e 6,393 e 8,216 da média do treino. O CatBoost fica à frente da baseline por UF, ao contrário do desenvolvimento, em que seu MAE médio (5,947) ficou ligeiramente acima do da baseline por UF (5,937). O MAE do teste é menor que o do desenvolvimento para todos os modelos listados, o que é compatível com prazos prometidos mais curtos nos meses de teste, mas essa leitura não foi testada.

Como a regressão não teve candidato e o teste não tem IC, a ordenação não seleciona modelo e o resultado não altera a decisão de P-04. O contraste com o desenvolvimento mostra apenas que a posição relativa dos modelos mudou entre os períodos, e diferenças pequenas não devem ser interpretadas.

## 8. F2: análise complementar pós-hoc e exploratória

O F2 foi desenhado depois de o período de teste do F1 ser observado, a partir de três sinais vistos nele: o prazo prometido, sozinho, foi um preditor forte; o CatBoost superestimou o risco; e as taxas históricas da rede mudaram. Por isso, qualquer resultado do F2 pode refletir ajuste ao próprio período, e o reuso adaptativo de um mesmo período de teste invalida a leitura confirmatória (Dwork et al., 2015). O F2 não confirma nem refuta o F1 e serve apenas para formular hipóteses a serem testadas em dados posteriores.

O desenho simula uma implantação: o modelo entra em uso em 1º de julho de 2018 e é atualizado em 2018-08-01, usando somente os rótulos já conhecidos nessa data, isto é, de pedidos entregues estritamente antes dela. A avaliação cobre as compras de agosto de 2018, e o IC vem de bootstrap por blocos diários da data de compra (Künsch, 1989), porque agosto tem poucas semanas. As variantes formam uma lista fechada, e todas são reportadas.

| Código | Variante | Mecanismo testado |
|---|---|---|
| A | Modelo do F1 congelado, sem atualização | Referência |
| B | Mesma configuração do F1, retreinada com todos os rótulos conhecidos | Retreino mensal |
| C | B com pesos de recência, meia-vida de 90 dias | Adaptação a mudança de regime (Gama et al., 2014) |
| D | B com recalibração isotônica nos rótulos entregues nos últimos 30 dias | Correção de mudança de prevalência |
| E | B com a feature `promise_vs_recent_median` | Captura de mudança na política de promessa |

As tabelas a seguir mostram a procedência do resultado, o estado de cada variante e os conjuntos usados em cada uma.

In [16]:
display(f2["provenance"])
display(f2["variants"])
display(f2["populations"])

,version,post_hoc,reading,production_eligible,created_at,fingerprint,update_date,eval_start,eval_end_exclusive,known_label_orders,eval_orders,eval_blocks,gate_passed,gate_failed,n_boot,seed,f1_model_path,f1_model_sha256
0,F2,True,exploratório,False,2026-11-14T12:02:47,a6fd373f862c7c6a68d92626992dd6bdf08f00b8a26a11...,2018-08-01,2018-08-01,2018-09-01,88097,6351,29,"C,E",D,1000,42,d:\Projetos\E-commerce-Delivery-Network-&-SLA-...,ace91068a027578a40abd579fa968938b332a74c5f484a...


,code,variant,gated,scored,status
0,A,A_f1_frozen,False,True,avaliada
1,B,B_retrain,False,True,avaliada
2,C,C_recency,True,True,avaliada
3,D,D_recalibration,True,False,reprovada no gate
4,E,E_promise_feature,True,True,avaliada


,variant,role,orders,breaches,prevalence,first_purchase,last_purchase,first_delivery,last_delivery
0,A_f1_frozen,"train (F1, frozen)",77867,5862,0.075282,2016-09-15 12:16:38,2018-05-31 23:51:24,2016-10-11 13:46:32,2018-10-11 16:41:14
1,B_retrain,train (known labels),88097,5922,0.067221,2016-09-15 12:16:38,2018-07-30 13:26:53,2016-10-11 13:46:32,2018-07-31 23:44:41
2,C_recency,train (known labels),88097,5922,0.067221,2016-09-15 12:16:38,2018-07-30 13:26:53,2016-10-11 13:46:32,2018-07-31 23:44:41
3,E_promise_feature,train (known labels),88097,5922,0.067221,2016-09-15 12:16:38,2018-07-30 13:26:53,2016-10-11 13:46:32,2018-07-31 23:44:41
4,baselines,train (known labels),88097,5922,0.067221,2016-09-15 12:16:38,2018-07-30 13:26:53,2016-10-11 13:46:32,2018-07-31 23:44:41
5,all,evaluation (August purchases),6351,393,0.061880,2018-08-01 00:02:17,2018-08-29 15:00:37,2018-08-02 15:42:36,2018-09-25 00:47:25


### Insight

A procedência marca o F2 como pós-hoc, com leitura exploratória, `production_eligible` falso e data de atualização em 2018-08-01. O treino das variantes retreinadas (B, C e E) e das baselines tem 88.097 pedidos com rótulo conhecido e prevalência de 6,7% (5.922 violações), contra 77.867 pedidos e 7,5% (5.862 violações) do modelo congelado do F1 (A). A avaliação usa 6.351 pedidos de agosto (393 violações; 6,2%) em 29 blocos diários. Foram avaliadas as variantes A, B, C e E, e a D foi reprovada no gate, que C e E passaram.

O treino de A inclui entregas até 2018-10-11, posteriores à data de atualização, enquanto o das demais termina em 2018-07-31, e essa diferença de informação pode afetar a comparação de A com as retreinadas. Os 29 blocos diários são poucos e têm correlação interna, o que limita a precisão dos intervalos.

Uma condição necessária da argumentação do F2 é que o regime em que a promessa discrimina o atraso não existia no desenvolvimento. A tabela mostra, por mês de compra, a mediana do prazo prometido e, só para os meses anteriores a julho de 2018, a prevalência e a AUC da regra do prazo prometido negativo. Para os meses seguintes, os rótulos não são lidos. O gráfico mostra a mediana em barras e a AUC em linha, com a referência do acaso.

In [17]:
from ecom_delivery_sla import promise_regime_table
from ecom_delivery_sla_viz import plot_promise_regime

regime = promise_regime_table(supervised)
display(regime)
plot_promise_regime(regime, language=LANG).show()

,purchase_month,orders,promise_median,labels_used,breach_prevalence,auc_negative_promise
0,2016-09,1,18.488449,True,1.000000,NaN
1,2016-10,265,54.152095,True,0.007547,0.714829
2,2016-12,1,26.030012,True,0.000000,NaN
3,2017-01,750,38.982905,True,0.029333,0.618007
4,2017-02,1653,30.176470,True,0.029643,0.614586
5,2017-03,2546,22.585220,True,0.045562,0.403697
6,2017-04,2303,25.470544,True,0.065567,0.570412
7,2017-05,3545,23.335255,True,0.029901,0.631309
8,2017-06,3135,23.419803,True,0.030303,0.527798
9,2017-07,3872,23.012668,True,0.027893,0.424702


### Insight

De 2017-01 a 2018-06, a AUC da regra do prazo prometido negativo ficou entre 0,40 e 0,63, sem tendência, ou seja, em torno do acaso (0,5): foi 0,404 em 2017-03, 0,631 em 2017-05 e 0,578 em 2018-06. A mediana do prazo prometido variou entre 21,3 e 27,4 dias de 2017-03 a 2018-06 e caiu para 19,3 dias em julho e 13,4 dias em agosto de 2018. O regime em que a promessa discrimina o atraso, portanto, não aparece nos rótulos de desenvolvimento, e nenhuma variante baseada nesse comportamento pode ser aprovada por eles. As AUC de 0,725 no F1 e de 0,777 em agosto, vistas adiante, foram obtidas nos meses em que a mediana já havia caído.

Os meses sem AUC têm rótulos não lidos nesta verificação, e os primeiros meses da série, como 2016-10 com 265 pedidos e AUC de 0,715, têm poucos pedidos e não devem ser lidos como regime. A coincidência no tempo entre a queda da mediana e a AUC alta é compatível com uma mudança na política de promessa, mas não demonstra a causa.

Na data de atualização, parte dos pedidos recentes ainda não tinha sido entregue e, portanto, não tinha rótulo. A tabela mostra, para as compras de junho e de julho de 2018, quantos pedidos tinham rótulo conhecido e a prevalência de violações em cada grupo. O gráfico empilha os pedidos com rótulo conhecido e desconhecido em cada mês e traz a prevalência de cada grupo. Esse é o problema do feedback atrasado (Chapelle, 2014): os desfechos mais lentos são os últimos a chegar.

In [18]:
from ecom_delivery_sla_viz import plot_label_availability

display(f2["label_availability"])
plot_label_availability(f2["label_availability"], language=LANG).show()

,purchase_month,as_of,orders,known_labels,share_known,prevalence_known,unknown_labels,prevalence_unknown
0,2018-06,2018-08-01,6096,6070,0.995735,0.008402,26,0.769231
1,2018-07,2018-08-01,6156,4164,0.676413,0.003122,1992,0.097892


### Insight

Em junho de 2018, 99,6% dos 6.096 pedidos tinham rótulo conhecido na data de atualização, com prevalência de 0,8%, enquanto os 26 pedidos ainda sem rótulo tinham prevalência de 76,9%. Em julho, apenas 67,6% dos 6.156 pedidos tinham rótulo conhecido, com prevalência de 0,3%, contra 9,8% entre os 1.992 pedidos ainda sem rótulo. A prevalência do conjunto de julho é de 3,4%, muito acima da dos rótulos conhecidos, de modo que os rótulos recentes disponíveis subestimam a prevalência do mês e favorecem entregas rápidas.

Esse viés de rótulos lentos entra no treino de todas as variantes retreinadas e é mais intenso na janela recente usada na recalibração da variante D. A tabela é descritiva e não corrige o treino.

Antes de qualquer contato com agosto, os mecanismos das variantes C, D e E foram aplicados nos folds de desenvolvimento e comparados com a configuração do F1. Uma variante seguia para agosto apenas se fosse não inferior: AP média de pelo menos 95% da AP da configuração do F1 e ECE médio no máximo 0,01 acima. O gate mede não inferioridade, e não ganho. A primeira tabela resume o gate por variante, a segunda traz as métricas por fold e o gráfico mostra a AP de cada variante por fold, com a configuração do F1 tracejada.

In [19]:
display(f2["gate_summary"])
display(f2["gate_folds"])
plot_metric_by_fold(
    f2["gate_folds"],
    fold_col="fold",
    model_col="variant",
    metric_col="average_precision",
    reference="F1_config",
    language=LANG,
).show()

,variant,n_folds,mean_ap,reference_mean_ap,ap_ratio,ap_ratio_threshold,passes_ap,mean_ece,reference_mean_ece,ece_difference,ece_margin,passes_ece,passed,status
0,C_recency,9,0.170719,0.174999,0.975542,0.95,True,0.058680,0.057619,0.001061,0.01,True,True,avaliada
1,D_recalibration,9,0.145307,0.174999,0.830333,0.95,False,0.065182,0.057619,0.007563,0.01,True,False,reprovada no gate
2,E_promise_feature,9,0.169836,0.174999,0.970496,0.95,True,0.057674,0.057619,0.000056,0.01,True,True,avaliada


,fold,variant,update_date,n_validation,n_fit,n_calibration,average_precision,ece_10bins,brier
0,2017-10,F1_config,2017-09-01,4478,22264,0,0.084222,0.012413,0.039591
1,2017-10,C_recency,2017-09-01,4478,22264,0,0.078871,0.017704,0.039853
2,2017-10,D_recalibration,2017-09-01,4478,18232,4032,0.062233,0.013493,0.039909
3,2017-10,E_promise_feature,2017-09-01,4478,22264,0,0.075715,0.015709,0.039834
4,2017-11,F1_config,2017-10-02,7288,26538,0,0.207368,0.087310,0.113753
5,2017-11,C_recency,2017-10-02,7288,26538,0,0.209523,0.092434,0.114730
6,2017-11,D_recalibration,2017-10-02,7288,22417,4121,0.172526,0.081899,0.113618
7,2017-11,E_promise_feature,2017-10-02,7288,26538,0,0.186419,0.088827,0.114706
8,2017-12,F1_config,2017-11-01,5513,30892,0,0.122211,0.043552,0.069908
9,2017-12,C_recency,2017-11-01,5513,30892,0,0.120922,0.046552,0.070176


### Insight

As variantes C e E passaram no gate e a D foi reprovada. A razão entre a AP média e a da configuração do F1 (0,175) foi de 0,976 para C (AP média de 0,171) e de 0,970 para E (0,170), acima do limite de 0,95, e a diferença de ECE foi de 0,001 e de 0,000, dentro da margem de 0,01. A D teve razão de 0,830 (AP média de 0,145) e foi reprovada pela AP, pois seu ECE médio (0,065, diferença de 0,008) ficaria dentro da margem. Em cada um dos 9 folds, a AP da D foi menor que a da configuração do F1. Como o gate mede não inferioridade, C e E, com AP média ligeiramente inferior à da referência, passam sem demonstrar ganho.

Nos folds, a janela de recalibração é definida pela data de compra e não pela data de entrega, de modo que o gate não reproduz o viés de rótulos lentos da atualização real, e o resultado da D no gate pode não refletir o que ocorreria em agosto. A variante reprovada não foi reavaliada com outros limiares depois de conhecido o resultado de agosto.

A tabela de métricas traz a estimativa pontual e o IC 95% por blocos diários das variantes avaliadas e das baselines, e a variante reprovada no gate aparece sem estimativa. A segunda tabela traz a diferença pareada de AP de cada variante contra a regressão logística com prazo e UF e contra a regra do prazo prometido negativo. Nenhum rótulo de confirmado ou não confirmado é aplicado: a leitura é sempre exploratória. O gráfico mostra a AP de cada modelo com o IC e, ao lado, as diferenças pareadas em relação à linha de zero.

In [20]:
metrics_f2 = f2["classification_metrics"]
display(metrics_f2[["model", "metric", "estimate", "ci_low", "ci_high", "status"]])
sampled = metrics_f2.dropna(subset=["n_boot"])
print(
    LABELS["discarded"].format(
        n=int(sampled["n_discarded"].iloc[0]),
        total=int(sampled["n_boot"].iloc[0]),
    )
)

paired_f2 = f2["paired_difference"]
display(
    paired_f2[
        [
            "model",
            "reference",
            "estimate_model",
            "estimate_reference",
            "difference",
            "ci_low",
            "ci_high",
            "difference_sign",
            "ci_excludes_zero",
            "reading",
            "status",
        ]
    ]
)
plot_metric_intervals(metrics_f2, metric="average_precision", paired=paired_f2, language=LANG).show()

,model,metric,estimate,ci_low,ci_high,status
0,A_f1_frozen,average_precision,0.069026,0.054170,0.083777,avaliada
1,A_f1_frozen,roc_auc,0.555992,0.516885,0.599745,avaliada
2,A_f1_frozen,brier,0.063841,0.048610,0.079037,avaliada
3,A_f1_frozen,ece_10bins,0.041298,0.034067,0.056659,avaliada
4,A_f1_frozen,precision_at_5pct,0.081761,0.053690,0.107878,avaliada
5,A_f1_frozen,recall_at_5pct,0.066158,0.042040,0.096222,avaliada
6,A_f1_frozen,precision_at_10pct,0.066038,0.046233,0.088289,avaliada
7,A_f1_frozen,recall_at_10pct,0.106870,0.073115,0.157310,avaliada
8,B_retrain,average_precision,0.068454,0.053698,0.082890,avaliada
9,B_retrain,roc_auc,0.556883,0.517592,0.600773,avaliada


Reamostragens descartadas por conter uma só classe: 0 de 1000


,model,reference,estimate_model,estimate_reference,difference,ci_low,ci_high,difference_sign,ci_excludes_zero,reading,status
0,A_f1_frozen,baseline_logistic_promise_plus_state,0.069026,0.093841,-0.024816,-0.043786,-0.010757,negative,True,exploratório,avaliada
1,A_f1_frozen,baseline_negative_promised_lead_days,0.069026,0.178748,-0.109723,-0.208806,-0.039556,negative,True,exploratório,avaliada
2,B_retrain,baseline_logistic_promise_plus_state,0.068454,0.093841,-0.025388,-0.044839,-0.009873,negative,True,exploratório,avaliada
3,B_retrain,baseline_negative_promised_lead_days,0.068454,0.178748,-0.110295,-0.209815,-0.040177,negative,True,exploratório,avaliada
4,C_recency,baseline_logistic_promise_plus_state,0.067454,0.093841,-0.026387,-0.046548,-0.011402,negative,True,exploratório,avaliada
5,C_recency,baseline_negative_promised_lead_days,0.067454,0.178748,-0.111294,-0.209972,-0.041853,negative,True,exploratório,avaliada
6,E_promise_feature,baseline_logistic_promise_plus_state,0.069296,0.093841,-0.024545,-0.043787,-0.009941,negative,True,exploratório,avaliada
7,E_promise_feature,baseline_negative_promised_lead_days,0.069296,0.178748,-0.109452,-0.209801,-0.039473,negative,True,exploratório,avaliada
8,D_recalibration,baseline_logistic_promise_plus_state,NaN,NaN,NaN,NaN,NaN,not evaluated,False,exploratório,reprovada no gate
9,D_recalibration,baseline_negative_promised_lead_days,NaN,NaN,NaN,NaN,NaN,not evaluated,False,exploratório,reprovada no gate


### Insight

As variantes avaliadas são praticamente indistinguíveis: AP de 0,069 (A, IC de 0,054 a 0,084), 0,068 (B, 0,054 a 0,083), 0,067 (C, 0,052 a 0,082) e 0,069 (E, 0,055 a 0,083), com AUC de 0,556, 0,557, 0,542 e 0,565. A comparação entre A e B, que isola o efeito de atualizar o modelo, mostra AP praticamente igual (0,069 e 0,068), e a feature de promessa de E também deixa a AP em 0,069. As variantes ficam acima da baseline de prevalência (0,062; IC de 0,044 a 0,079) apenas na estimativa pontual, com intervalos sobrepostos, e abaixo da regressão logística (0,094; 0,071 a 0,121) e da regra do prazo prometido negativo (0,179; 0,105 a 0,284). As diferenças pareadas contra a logística são de −0,025 (A), −0,025 (B), −0,026 (C) e −0,025 (E), e contra a regra, de −0,110 (A), −0,110 (B), −0,111 (C) e −0,109 (E), todas com IC que exclui zero.

Esses sinais são tamanhos de efeito exploratórios, sem correção para múltiplas comparações, calculados sobre 29 blocos diários correlacionados de um período já consultado. A variante D não tem estimativa, e o F2 não confirma nem refuta o F1.

A curva de confiabilidade de agosto compara a probabilidade média prevista com a taxa observada em cada faixa, para as variantes avaliadas e as baselines probabilísticas. A tabela de precisão e recall nos 5% e 10% de maior risco resume o uso do escore para priorização.

In [21]:
reliability_f2 = f2["reliability"]
display(reliability_f2)
plot_reliability_curves(reliability_f2, language=LANG).show()
display(f2["top_k"])

,model,bin,bin_lower,bin_upper,n,mean_predicted,observed_rate
0,A_f1_frozen,1,0.0,0.1,4521,0.053573,0.061712
1,A_f1_frozen,2,0.1,0.2,1213,0.139892,0.060181
2,A_f1_frozen,3,0.2,0.3,461,0.242664,0.060738
3,A_f1_frozen,4,0.3,0.4,123,0.339002,0.089431
4,A_f1_frozen,5,0.4,0.5,24,0.431044,0.083333
5,A_f1_frozen,6,0.5,0.6,2,0.558429,0.000000
6,A_f1_frozen,7,0.6,0.7,4,0.633337,0.000000
7,A_f1_frozen,8,0.7,0.8,2,0.718727,0.000000
8,A_f1_frozen,9,0.8,0.9,1,0.800651,0.000000
9,A_f1_frozen,10,0.9,1.0,0,NaN,NaN


,model,top_fraction,precision,precision_ci_low,precision_ci_high,recall,recall_ci_low,recall_ci_high
0,A_f1_frozen,0.05,0.081761,0.053690,0.107878,0.066158,0.042040,0.096222
1,A_f1_frozen,0.10,0.066038,0.046233,0.088289,0.106870,0.073115,0.157310
2,B_retrain,0.05,0.075472,0.041973,0.101855,0.061069,0.033043,0.091252
3,B_retrain,0.10,0.064465,0.043829,0.088381,0.104326,0.073491,0.149655
4,C_recency,0.05,0.072327,0.040397,0.101984,0.058524,0.034085,0.085724
5,C_recency,0.10,0.069182,0.042699,0.092784,0.111959,0.072577,0.156874
6,E_promise_feature,0.05,0.075472,0.042207,0.103787,0.061069,0.032035,0.098046
7,E_promise_feature,0.10,0.067610,0.044630,0.089724,0.109415,0.070030,0.158787
8,baseline_prevalence,0.05,0.062893,0.010134,0.141540,0.050891,0.009091,0.111671
9,baseline_prevalence,0.10,0.056604,0.018122,0.117559,0.091603,0.032150,0.181626


### Insight

O erro de calibração da variante A tem o sentido de superestimação, como no F1: de 0,1 a 0,2, de 0,2 a 0,3 e de 0,3 a 0,4, a probabilidade média prevista é 0,140, 0,243 e 0,339, contra taxas observadas de 0,060, 0,061 e 0,089 (1.213, 461 e 123 pedidos). As variantes retreinadas têm o sentido oposto: B concentra 6.302 dos 6.351 pedidos na primeira faixa, com média prevista de 0,021 contra 0,062 observada, e C (6.346 pedidos; 0,013 contra 0,062) e E (6.298 pedidos; 0,023 contra 0,062) fazem o mesmo, ou seja, subestimam o risco. O ECE não reflete essa troca de direção: é 0,041 em A, 0,041 em B, 0,049 em C e 0,039 em E, contra 0,036 na regressão logística e 0,005 na baseline de prevalência.

Uma mudança de direção do erro, sem redução do ECE, é compatível com o viés de rótulos lentos da seção anterior, em que os rótulos conhecidos de julho tinham prevalência de 0,3%, mas essa explicação não foi testada. Como as variantes retreinadas concentram quase todos os pedidos em uma só faixa, o ECE depende quase só do desvio nessa faixa e diz pouco sobre a discriminação.

## 9. O que os dados sustentam e o que não sustentam

A tabela a seguir reúne em um só lugar as estimativas centrais das três etapas: o ganho de desenvolvimento do modelo candidato, as estimativas do F1 com seus intervalos e a diferença pareada com a leitura pré-registrada, e as estimativas do F2 com a leitura exploratória. Todos os valores são lidos das tabelas gravadas já exibidas nas seções anteriores.

In [22]:
from ecom_delivery_sla_viz.report import evidence_table

display(
    evidence_table(
        classification_summary,
        metrics_f1,
        paired_f1,
        metrics_f2,
        paired_f2,
        language=LANG,
    )
)

,stage,item,estimate,ci_low,ci_high,note
0,Desenvolvimento (folds de 2017-10 a 2018-06),AP média do CatBoost,0.174999,NaN,NaN,
1,Desenvolvimento (folds de 2017-10 a 2018-06),AP média da melhor baseline (baseline_logistic...,0.153627,NaN,NaN,
2,Desenvolvimento (folds de 2017-10 a 2018-06),Ganho relativo de AP sobre a melhor baseline,0.139115,NaN,NaN,8 de 9 folds vencidos
3,F1 (oficial),AP: CatBoost,0.081983,0.055835,0.133698,
4,F1 (oficial),AP: baseline_prevalence,0.048053,0.027216,0.068543,
5,F1 (oficial),AP: baseline_negative_promised_lead_days,0.141621,0.043965,0.231634,
6,F1 (oficial),AP: baseline_logistic_promise_plus_state,0.084250,0.056228,0.115516,
7,F1 (oficial),Diferença pareada de AP: CatBoost menos baseli...,-0.002267,-0.032492,0.063988,inconclusivo
8,"F2 (pós-hoc, exploratório)",AP: A_f1_frozen,0.069026,0.054170,0.083777,
9,"F2 (pós-hoc, exploratório)",AP: B_retrain,0.068454,0.053698,0.082890,


### Insight

Nas três etapas, o CatBoost não se separa da baseline logística fora do desenvolvimento e fica abaixo da regra do prazo prometido. No desenvolvimento, a AP média é 0,175 contra 0,154, um ganho de 13,9% em 8 de 9 folds. No F1, a AP é 0,082 contra 0,084 da logística, com diferença pareada de −0,002 (IC de −0,032 a +0,064), inconclusiva, enquanto a regra do prazo prometido tem AP de 0,142. No F2, as variantes A, B, C e E têm AP de 0,069, 0,068, 0,067 e 0,069, contra 0,094 da logística e 0,179 da regra, com diferenças pareadas de cerca de −0,025 contra a logística e IC que exclui zero, e a D foi reprovada no gate. A AP da regra do prazo prometido passou de 0,078 no desenvolvimento para 0,142 no F1 e 0,179 no F2. O ponto central é o contraste entre o ganho de 13,9% do desenvolvimento e a diferença pareada próxima de zero do F1, cujo IC é compatível tanto com perda quanto com ganho.

As estimativas do F2 devem ser lidas como exploratórias: o F2 não confirma nem refuta o F1, não tem correção para múltiplas comparações e usa um período já consultado. A vantagem do desenvolvimento foi medida em folds anteriores a julho de 2018, em que a regra do prazo prometido tinha AUC em torno do acaso, e a transferência dessa vantagem para o regime recente de promessa não foi demonstrada.

Os dados sustentam, em primeiro lugar, que em dados anteriores a julho de 2018 o CatBoost superou a melhor baseline de forma consistente entre os folds e foi o único candidato pelo critério pré-registrado, enquanto a regressão do atraso não teve candidato. Sustentam também que, no período de teste do F1, essa vantagem não se confirmou: o resultado oficial foi inconclusivo, com intervalos largos e sobreposição com a baseline logística. Sustentam ainda que a regra simples do prazo prometido foi uma referência forte nos períodos recentes, e que a calibração do modelo depende do regime de dados em que foi treinado.

Os dados não sustentam que o CatBoost supere as baselines fora do período de desenvolvimento, nem que ele deva ser usado operacionalmente. Não sustentam que algum mecanismo de atualização do F2 seja melhor que o modelo congelado: o F2 é pós-hoc, compara várias variantes sem correção para múltiplas comparações e usa um período já consultado. Não sustentam a explicação do viés de rótulos lentos como causa da piora, que é compatível com as tabelas, mas não foi testada. Também não sustentam relações causais entre rota, vendedor, promessa e atraso, porque todas as análises são preditivas ou descritivas. Por essas razões, `production_eligible` permanece `false`.

## 10. Limitações

A primeira limitação é a censura à direita: pedidos sem entrega no fim da série foram excluídos, de modo que a população de teste é condicionada à entrega observada, e o cenário que os trata como violação é apenas um limite superior descritivo. A segunda é o drift do prazo prometido, cuja distribuição muda entre desenvolvimento e teste, de modo que qualquer vantagem ou desvantagem pode refletir essa mudança. A terceira é o viés de atraso de rótulo no histórico da rede: o desfecho de um pedido só entra no histórico quando a entrega é conhecida, e as taxas recentes tendem a subestimar atrasos.

A quarta limitação é que o período de teste não é um holdout virgem: versões anteriores do projeto, com outro alvo e outras features, já o haviam consultado, e o F2 foi desenhado depois de o F1 ser observado. A quinta é a largura dos intervalos, porque o teste tem poucas semanas e o F2 tem poucos dias, com correlação dentro de cada bloco. A sexta é a ausência de correção para múltiplas comparações no F2. A sétima é o escopo dos dados: uma única plataforma, um intervalo de dois anos, sem informações operacionais como capacidade, tráfego ou rota real. Por fim, o pré-processamento das variáveis categóricas não foi alterado depois da abertura do teste, e a importação do pacote por manipulação de `sys.path` nos notebooks permanece como limitação conhecida de reprodução.

## 11. Hipóteses para um próximo ciclo

As hipóteses a seguir nasceram das observações descritas neste relatório e só podem ser testadas em dados posteriores a agosto de 2018, que não existem no conjunto Olist. A primeira é que a atualização do modelo deve usar apenas coortes de desfecho maduro ou tratar os pedidos em trânsito como censurados, em um modelo de sobrevivência (Kaplan e Meier, 1958), para evitar o viés de rótulos lentos. A segunda é que a regra do prazo prometido pode servir de alerta de mudança de política de promessa, e não como modelo. A terceira é que o monitoramento contínuo da prevalência e do PSI das features de rede indicaria a necessidade de recalibração antes da degradação do desempenho. Cada hipótese exige um protocolo próprio, registrado antes da avaliação, e nenhuma delas autoriza uso operacional.

## Referências

Akiba, T., Sano, S., Yanase, T., Ohta, T. e Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. *Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining*, 2623–2631. https://doi.org/10.1145/3292500.3330701

Bergmeir, C. e Benítez, J. M. (2012). On the use of cross-validation for time series predictor evaluation. *Information Sciences*, 191, 192–213. https://doi.org/10.1016/j.ins.2011.12.028

Brier, G. W. (1950). Verification of forecasts expressed in terms of probability. *Monthly Weather Review*, 78(1), 1–3.

Chapelle, O. (2014). Modeling delayed feedback in display advertising. *Proceedings of the 20th ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 1097–1105. https://doi.org/10.1145/2623330.2623634

Davis, J. e Goadrich, M. (2006). The relationship between precision-recall and ROC curves. *Proceedings of the 23rd International Conference on Machine Learning*, 233–240. https://doi.org/10.1145/1143844.1143874

DeGroot, M. H. e Fienberg, S. E. (1983). The comparison and evaluation of forecasters. *The Statistician*, 32(1/2), 12–22. https://doi.org/10.2307/2987588

Dwork, C., Feldman, V., Hardt, M., Pitassi, T., Reingold, O. e Roth, A. (2015). The reusable holdout: preserving validity in adaptive data analysis. *Science*, 349(6248), 636–638. https://doi.org/10.1126/science.aaa9375

Gama, J., Žliobaitė, I., Bifet, A., Pechenizkiy, M. e Bouchachia, A. (2014). A survey on concept drift adaptation. *ACM Computing Surveys*, 46(4), artigo 44. https://doi.org/10.1145/2523813

Kaplan, E. L. e Meier, P. (1958). Nonparametric estimation from incomplete observations. *Journal of the American Statistical Association*, 53(282), 457–481. https://doi.org/10.1080/01621459.1958.10501452

Klein, J. P. e Moeschberger, M. L. (2003). *Survival Analysis: Techniques for Censored and Truncated Data* (2ª ed.). Springer.

Künsch, H. R. (1989). The jackknife and the bootstrap for general stationary observations. *Annals of Statistics*, 17(3), 1217–1241. https://doi.org/10.1214/aos/1176347265

Naeini, M. P., Cooper, G. F. e Hauskrecht, M. (2015). Obtaining well calibrated probabilities using Bayesian binning. *Proceedings of the 29th AAAI Conference on Artificial Intelligence*, 2901–2907. https://doi.org/10.1609/aaai.v29i1.9602

Niculescu-Mizil, A. e Caruana, R. (2005). Predicting good probabilities with supervised learning. *Proceedings of the 22nd International Conference on Machine Learning*, 625–632. https://doi.org/10.1145/1102351.1102430

Nosek, B. A., Ebersole, C. R., DeHaven, A. C. e Mellor, D. T. (2018). The preregistration revolution. *Proceedings of the National Academy of Sciences*, 115(11), 2600–2606. https://doi.org/10.1073/pnas.1708274114

Olist (2018). *Brazilian E-Commerce Public Dataset by Olist*. Kaggle. https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

Pedregosa, F. et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.

Prokhorenkova, L., Gusev, G., Vorobev, A., Dorogush, A. V. e Gulin, A. (2018). CatBoost: unbiased boosting with categorical features. *Advances in Neural Information Processing Systems 31*.

Siddiqi, N. (2006). *Credit Risk Scorecards: Developing and Implementing Intelligent Credit Scoring*. Wiley.

Wilson, E. B. (1927). Probable inference, the law of succession, and statistical inference. *Journal of the American Statistical Association*, 22(158), 209–212. https://doi.org/10.1080/01621459.1927.10502953

Zadrozny, B. e Elkan, C. (2002). Transforming classifier scores into accurate multiclass probability estimates. *Proceedings of the 8th ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 694–699. https://doi.org/10.1145/775047.775151